In [1]:
# print("Hellow!")

In [2]:
import os
import json
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from google import genai
from google.genai import types
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

In [4]:
from ingest import load_faq_data, build_index

In [5]:
from rag_helper import RAGBase

In [6]:
model='gemini-2.5-flash'

In [7]:
documents = load_faq_data()
index = build_index(documents)

In [8]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index = index,
    llm_client = client,
    instructions = instructions
)

In [8]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

To run Ollama locally, follow these steps based on the provided context:

### 1. Install Ollama
First, download and install Ollama for your operating system by visiting [https://ollama.com/download](https://ollama.com/download):
* **macOS**: Download and install the `.pkg` file.
* **Windows**: Download and install the `.msi` file.
* **Linux**: Run the following command in your terminal:
  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```

### 2. Run the Model
Once installed, open a terminal and run the following command to download the LLaMA 3 model (~4GB), start it locally, and open a chat interface:
```bash
ollama run llama3
```

### 3. Test the Local Server
To verify that the Ollama local server is running, execute:
```bash
curl http://localhost:11434
```
You should receive a JSON response listing the models.

### 4. (Optional) Run via Python
If you want to interact with it using Python, install the client:
```bash
pip install ollama
```
Then run a minimal Python script:

In [ ]:
messages = [
    types.Content(
        role="user",
        parts=[types.Part.from_text(text="I just discovered the course. Can I join it?")]
    )
]

config = types.GenerateContentConfig(
    temperature=0.2, # Controls randomness (lower is more deterministic)
    max_output_tokens=800   
)

response = client.models.generate_content(
        model=model,
        contents=messages,
        config=config
)

# response.text

"That's great you've discovered it! To give you an accurate answer, I'll need a little more information about the specific course you're"

In [10]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [28]:
# search_tool = types.Tool(
#     function_declarations=[
#         types.FunctionDeclaration(
#             name="search",
#             description="Search the FAQ database for entries matching the given query.",
#             parameters=types.Schema(
#                 type=types.Type.OBJECT,
#                 properties={
#                     "query": types.Schema(
#                         type=types.Type.STRING,
#                         description="Search query text to look up in the course FAQ."
#                     )
#                 },
#                 required=["query"]
#             )
#         )
#     ]
# )

In [11]:
search_tool = types.FunctionDeclaration(
    name="search",
    description="Search the course database for relevant context and answers.",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "query": types.Schema(
                type=types.Type.STRING,
                description="The search query text based on user question.",
            )
        },
        required=["query"],
    )
)    

In [12]:
tool_config = types.GenerateContentConfig(
    tools=[types.Tool(
        function_declarations=[search_tool]
        )]
)

In [27]:
# tool_config = types.GenerateContentConfig(
#     tools=[search_tool],
#     temperature=0.2
# )

In [13]:
response = client.models.generate_content(
    model=model,
    contents=messages,
    config=tool_config # Pass tools here
)

In [19]:
response.text

In [14]:
call = response.function_calls[0]

In [18]:
print(call)

id=None args={'query': 'join course'} name='search' partial_args=None will_continue=None


In [15]:
arg = call.args
# print(f"Function: {call.name}")
# print(f"Arguments: {arg}")

In [ ]:
call.name

In [16]:
search_results = search(arg['query'])

In [17]:
results_string = json.dumps(search_results)

In [ ]:
print("Successfully retrieved search context!")
print(results_string)

[{"course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions.", "doc_id": "74eb249bbf"}, {"course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "When will the course be offered next?", "answer": "Summer 2027.", "doc_id": "bd31146b0e"}, {"course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "How should I start the course and follow the weekly workflow?", "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and 

In [18]:
messages.append(response.candidates[0].content)

In [35]:
# function_call_output = {
#     "type": "function_call_output",
#     'call_id': call.call_id,
#     'output': results_string
# }

In [19]:
function_response_part = types.Part.from_function_response(
        name=call.name,
        response={"result": results_string}
    )

In [20]:
messages.append(
        types.Content(
            role="tool",
            parts=[function_response_part]
        )
)

In [21]:
final_response = client.models.generate_content(
        model=model,
        contents=messages,
        config=tool_config
)

In [40]:
print(final_response.text)

Yes, you can join the LLM Zoomcamp course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.

Is this the course you were referring to, or were you interested in a different one?


In [41]:
# Extract token usage from metadata
usage = response.usage_metadata
prompt_tokens = usage.prompt_token_count
candidate_tokens = usage.candidates_token_count

# Print individual metrics
print(f"Prompt (Input) Tokens: {prompt_tokens}")
print(f"Candidates (Output) Tokens: {candidate_tokens}")
print(f"Total Tokens Used: {usage.total_token_count}")

Prompt (Input) Tokens: 60
Candidates (Output) Tokens: 13
Total Tokens Used: 205


In [42]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(prompt_tokens, candidate_tokens)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 1.68e-05


In [22]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [42]:
# 2. Define your local python search tool
def course_search(keywords: str) -> dict:
    """Search the course database for enrollment details and schedules.
    Args:
        keywords: The search terms to use.
    """
    print(f"Running Local Search Tool with keywords: '{keywords}'")
    return {"results": "Enrollment is OPEN for the next 48 hours. Course fee is $0. Prerequisite: basic Python knowledge."}

In [37]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [43]:
messages = [
    types.Content(
        role="user", 
        parts=[types.Part.from_text(text=question)]
    ),
    types.Content(
        role="developer", 
        parts=[types.Part.from_text(text=instructions)]
    ),
]

In [45]:
developer_text = next(
    (part.text for msg in messages for part in msg.parts 
     if msg.role == "developer"
    ), 
    None
)

filtered_messages = [msg for msg in messages if msg.role != "developer"]

In [46]:
config = types.GenerateContentConfig(
    system_instruction=developer_text,
    tools=[course_search], 
    temperature=0.2 
)

In [ ]:
while True:
    response = client.models.generate_content(
        model=model,
        contents=filtered_messages,
        config=config
    )
    
    # Track model's response in our history to keep context intact
    if response.candidates and response.candidates[0].content:
        filtered_messages.append(response.candidates[0].content)

    # Check if Gemini wants to call a function
    if response.function_calls:
        print("\n Gemini requested a tool execution.")
        
        for function_call in response.function_calls:
            if function_call.name == "course_search":
                # Extract arguments safely
                keywords_arg = function_call.args.get("keywords")
                
                # Execute your local Python function
                tool_output = course_search(keywords=keywords_arg)
                
                # Package the result using types.Part.from_function_response
                function_response_part = types.Part.from_function_response(
                    name=function_call.name,
                    response=tool_output
                )
                
                # Append tool results as a user role turnaround
                filtered_messages.append(
                    types.Content(role="user", parts=[function_response_part])
                )
        
        # Loop continues, sending the updated history containing search data back to Gemini
        print(" Sending search results back to Gemini for processing...\n")
        continue

    # If no more function calls were requested, print the final text answer and exit
    elif response.text:
        print("\n ASSISTANT FINAL ANSWER:")
        print(response.text)
        break
    else:
        print(f"Finished without text response. Reason: {response.candidates[0].finish_reason}")
        break

Running Local Search Tool with keywords: 'course enrollment'

✨ ASSISTANT FINAL ANSWER:
Yes, you can join the course! Enrollment is open for the next 48 hours and there is no course fee. However, you will need to have basic Python knowledge as a prerequisite.

Is there anything else you would like to explore about the course?
